In [4]:
import duckdb
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"

con = duckdb.connect("olist.duckdb")

tables = {
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers":   "olist_customers_dataset.csv",
    "payments":    "olist_order_payments_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
}

for name, file in tables.items():
    path = (DATA_DIR / file).as_posix() 
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv_auto('{path}')")

df = con.execute("SELECT COUNT(*) AS total_orders FROM orders").df()
print(df)


   total_orders
0         99441


In [5]:
df_revenue = con.execute("""
    SELECT
        DATE_TRUNC('month', order_purchase_timestamp::TIMESTAMP) AS month,
        COUNT(DISTINCT o.order_id) AS total_orders,   -- заказы, а не строки-позиции
        ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY month
    ORDER BY month
""").df()

print(df_revenue)


        month  total_orders     revenue
0  2016-09-01             1      143.46
1  2016-10-01           265    46490.66
2  2016-12-01             1       19.62
3  2017-01-01           750   127482.37
4  2017-02-01          1653   271239.32
5  2017-03-01          2546   414330.95
6  2017-04-01          2303   390812.40
7  2017-05-01          3546   566851.40
8  2017-06-01          3135   490050.37
9  2017-07-01          3872   566299.08
10 2017-08-01          4193   645832.36
11 2017-09-01          4150   701077.49
12 2017-10-01          4478   751117.01
13 2017-11-01          7289  1153364.20
14 2017-12-01          5513   843078.29
15 2018-01-01          7069  1077887.46
16 2018-02-01          6555   966168.41
17 2018-03-01          7003  1120598.24
18 2018-04-01          6798  1132878.93
19 2018-05-01          6749  1128774.52
20 2018-06-01          6099  1011978.29
21 2018-07-01          6159  1027807.28
22 2018-08-01          6351   985491.64


In [6]:
tables = ['orders', 'order_items', 'customers', 'payments', 'reviews']

for table in tables: 
    count = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table}: {count:,} rows')

orders: 99,441 rows
order_items: 112,650 rows
customers: 99,441 rows
payments: 103,886 rows
reviews: 99,224 rows


In [7]:
for table in tables:
    print(table)
    total = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    cols = con.execute(f"SELECT * FROM {table} LIMIT 0").df().columns

    for col in cols:
        nulls = con.execute(f"SELECT COUNT(*) FROM {table} WHERE {col} IS NULL").fetchone()[0]
        if nulls != 0:
            pct = round(nulls / total * 100, 1)
            print(f"  {col}: {nulls:,} nulls ({pct}%)")


orders
  order_approved_at: 160 nulls (0.2%)
  order_delivered_carrier_date: 1,783 nulls (1.8%)
  order_delivered_customer_date: 2,965 nulls (3.0%)
order_items
customers
payments
reviews
  review_comment_title: 87,656 nulls (88.3%)
  review_comment_message: 58,247 nulls (58.7%)


In [8]:
total = con.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
unique = con.execute("SELECT COUNT(DISTINCT order_id) FROM orders").fetchone()[0]

print(f"Total rows: {total:,}")
print(f"Unique order_ids: {unique:,}")
print(f"Duplicates: {total - unique:,}")


Total rows: 99,441
Unique order_ids: 99,441
Duplicates: 0


In [10]:
print('order items')
result = con.execute("""

    SELECT
                     
        COUNT(*) FILTER(WHERE price <= 0) AS zero_or_negative_price, 
        COUNT(*) FILTER(WHERE freight_value < 0) AS negative_freight,
        MIN(price) AS min_price, 
        MAX(price) AS max_price,
        ROUND(AVG(price), 2) AS avg_price
    FROM order_items
        
 """).df()
print(result)

order items
   zero_or_negative_price  negative_freight  min_price  max_price  avg_price
0                       0                 0       0.85     6735.0     120.65


In [11]:
print("payments")

result = con.execute("""
    SELECT 
        COUNT(*) FILTER(WHERE payment_value <= 0) AS zero_or_negative_payment, 
        MIN(payment_value) AS min_payment,
        MAX(payment_value) AS max_payment,
        ROUND(AVG(payment_value), 2) AS avg_payment
    FROM payments
""").df()
print(result)

payments
   zero_or_negative_payment  min_payment  max_payment  avg_payment
0                         9          0.0     13664.08        154.1


In [17]:
print("dates")

result = con.execute("""
    SELECT 
        COUNT(*) FILTER(WHERE order_purchase_timestamp > order_delivered_customer_date)  AS delivered_before_purchase, 
        COUNT(*) FILTER(WHERE order_purchase_timestamp > CURRENT_DATE) AS future_dates,
        MIN(order_purchase_timestamp) AS min_timestamp,
        MAX(order_purchase_timestamp) AS max_timestamp          
    FROM orders
""").df()

print(result)

dates
   delivered_before_purchase  future_dates       min_timestamp  \
0                          0             0 2016-09-04 21:15:19   

        max_timestamp  
0 2018-10-17 17:30:18  


In [21]:
print("status")

result = con.execute("""
    SELECT 
        order_status,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    
    FROM orders
    GROUP BY order_status
    ORDER BY count DESC
 """).df()

print(result)

status
  order_status  count   pct
0    delivered  96478  97.0
1      shipped   1107   1.1
2     canceled    625   0.6
3  unavailable    609   0.6
4     invoiced    314   0.3
5   processing    301   0.3
6      created      5   0.0
7     approved      2   0.0
